# Advanced Retrieval & Reranking: Making RAG Actually Work

Basic RAG pipelines-embed documents, embed a query, return top-k cosine neighbours-fail in predictable ways:

- **Vocabulary mismatch**: "myocardial infarction" ≠ "heart attack" in vector space unless the embedding model saw enough medical text.
- **Short query, long document**: a 5-word query rarely aligns well with a 500-word passage in the same embedding space.
- **Retrieval ≠ relevance**: approximate nearest-neighbour search optimises *speed*, not answer quality.
- **Lost in the middle**: even when the right chunk is retrieved, LLMs under-use context in the middle of a long prompt.
- **Single-query brittleness**: if the user's phrasing doesn't match the document's phrasing, retrieval fails silently.

This notebook works through the full stack of techniques that address these failure modes:

| Layer | Technique |
|---|---|
| Indexing | Better chunking, hierarchical indexes, graph overlays |
| Query | HyDE, multi-query, step-back, RAG-Fusion |
| Retrieval | Hybrid BM25+dense, SPLADE |
| Post-retrieval | Cross-encoder reranking, ColBERT, BGE |
| Architecture | Self-RAG, CRAG, RAPTOR, FLARE |
| Evaluation | RAGAS, TruLens, DeepEval |

## 1. Vector Databases Deep Dive

Choosing a vector database is a systems decision. Below is a concise map of the ecosystem.

### Major Vector Databases

| Database | Key Differentiator |
|---|---|
| **Milvus** | Distributed, GPU-accelerated, DiskANN for billion-scale, Attu UI |
| **Weaviate** | Schema-based, hybrid BM25+vector built-in, pluggable modules (text2vec, generative) |
| **LanceDB** | Serverless, columnar format (Apache Arrow/Lance), versioning, zero-copy reads |
| **Marqo** | End-to-end multimodal search, manages embeddings internally |
| **Redis Vector** | In-memory HNSW/FLAT indexes, sub-millisecond latency, familiar Redis API |
| **Elasticsearch/OpenSearch** | kNN with `dense_vector` field, mature ecosystem, hybrid BM25+kNN |
| **Vespa** | Streaming evaluation, multi-vector per document, rich ranking expressions |
| **ScaNN** (Google) | Anisotropic quantization, claimed 10-100× faster than brute force |
| **Annoy** | Approximate nearest neighbours with random projection trees, Spotify-built, read-optimised |
| **pgvector** | PostgreSQL extension, SQL joins over vectors, ACID transactions |

### Index Algorithm Comparison

| Algorithm | Search Complexity | Recall | Memory | Notes |
|---|---|---|---|---|
| **HNSW** | $O(\log n)$ | High | High | Default choice; great recall/speed tradeoff |
| **IVF** | $O(\sqrt{n})$ | Medium | Medium | Partition-based; needs training |
| **PQ / IVFPQ** | $O(\sqrt{n})$ + quant. | Lower | **Low** | Compress vectors; good for RAM-constrained |
| **Flat / Exact** | $O(n)$ | **Perfect** | High | Only viable for small corpora (<100 K) |
| **DiskANN** | $O(\log n)$ approx. | High | Low RAM | Graph stored on SSD; billion-scale |
| **ScaNN** | Sub-linear | High | Medium | Anisotropic quantization, batched queries |

HNSW builds a hierarchical graph where each node links to its closest neighbours at multiple granularity layers. Search starts at the top layer (few nodes) and greedily descends:

$$\text{complexity} = O(\log n) \text{ expected, } O(n) \text{ worst case}$$

IVF partitions the space into $k$ Voronoi cells. Query searches only the $n_{probe}$ nearest cells:

$$\text{complexity} \approx O\!\left(\sqrt{n}\right) \text{ when } n_{probe} \ll k$$

In [1]:
# Vector DB client comparison conceptual code (install each client separately)
# pip install pymilvus weaviate-client lancedb

# ── Milvus ────────────────────────────────────────────────────────────────────
MILVUS_EXAMPLE = '''
from pymilvus import connections, Collection, FieldSchema, CollectionSchema, DataType

connections.connect(host="localhost", port="19530")

fields = [
    FieldSchema(name="id",        dtype=DataType.INT64,         is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR,  dim=1536),
    FieldSchema(name="text",      dtype=DataType.VARCHAR,        max_length=4096),
]
schema = CollectionSchema(fields, description="RAG documents")
col    = Collection(name="docs", schema=schema)

# HNSW index
col.create_index("embedding", {"index_type": "HNSW", "metric_type": "COSINE",
                                "params": {"M": 16, "efConstruction": 200}})

# Search
col.load()
results = col.search([query_vec], "embedding", param={"metric_type": "COSINE", "ef": 64},
                      limit=10, output_fields=["text"])
'''

# ── Weaviate ──────────────────────────────────────────────────────────────────
WEAVIATE_EXAMPLE = '''
import weaviate

client = weaviate.connect_to_local()

# Create a collection with hybrid search enabled
from weaviate.classes.config import Configure, Property, DataType

client.collections.create(
    name="Document",
    vectorizer_config=Configure.Vectorizer.text2vec_openai(),
    properties=[Property(name="content", data_type=DataType.TEXT)],
)

docs = client.collections.get("Document")

# Hybrid search (BM25 + vector)
response = docs.query.hybrid(
    query="What is retrieval augmented generation?",
    alpha=0.75,   # 0 = pure BM25, 1 = pure vector
    limit=5
)
client.close()
'''

# ── LanceDB ───────────────────────────────────────────────────────────────────
LANCEDB_EXAMPLE = '''
import lancedb
import pandas as pd

db    = lancedb.connect("./my_lancedb")   # serverless just a directory
table = db.create_table("docs", data=[
    {"vector": [0.1]*1536, "text": "sample document", "source": "wiki"}
])

# Versioning every write creates a new version
table.add([{"vector": [0.2]*1536, "text": "new chunk", "source": "arxiv"}])
print(table.list_versions())  # [{"version": 1, ...}, {"version": 2, ...}]

# ANN search
results = table.search([0.1]*1536).limit(5).to_pandas()
'''

print("Milvus example:")
print(MILVUS_EXAMPLE)
print("\nWeaviate example:")
print(WEAVIATE_EXAMPLE)
print("\nLanceDB example:")
print(LANCEDB_EXAMPLE)

Milvus example:

from pymilvus import connections, Collection, FieldSchema, CollectionSchema, DataType

connections.connect(host="localhost", port="19530")

fields = [
    FieldSchema(name="id",        dtype=DataType.INT64,         is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR,  dim=1536),
    FieldSchema(name="text",      dtype=DataType.VARCHAR,        max_length=4096),
]
schema = CollectionSchema(fields, description="RAG documents")
col    = Collection(name="docs", schema=schema)

# HNSW index
col.create_index("embedding", {"index_type": "HNSW", "metric_type": "COSINE",
                                "params": {"M": 16, "efConstruction": 200}})

# Search
col.load()
results = col.search([query_vec], "embedding", param={"metric_type": "COSINE", "ef": 64},
                      limit=10, output_fields=["text"])


Weaviate example:

import weaviate

client = weaviate.connect_to_local()

# Create a collection with hybrid search enabled
from weaviate.clas

## 2. Chunking Strategies

Chunking is the most under-appreciated part of a RAG pipeline. The wrong chunk size degrades recall even with a perfect embedding model.

### Why Chunking Matters

- Too large → retrieval returns noisy context, token budget wasted
- Too small → retrieval loses surrounding context needed to answer the question
- Boundary placement → splitting mid-sentence destroys semantic coherence

### Strategy Taxonomy

| Strategy | Description | Best For |
|---|---|---|
| **Fixed-size** | `chunk_size=512`, `overlap=64` | Baseline; fast to implement |
| **Recursive character** | Split on `\n\n` → `\n` → ` ` → `""` | General text |
| **Semantic** | Embed sentences; split where cosine similarity drops | Dense, mixed-topic docs |
| **Agentic** | LLM identifies proposition boundaries | High-value precise Q&A |
| **Late chunking** | Encode full doc, then split token embeddings | Context-heavy passages |
| **Hierarchical** | Parent doc for context, child chunk for retrieval | Long documents |
| **Markdown header** | Split on `#`, `##`, `###` | Documentation, wikis |
| **Sentence-based** | Split on sentence boundaries | Conversational text |
| **Proposition-based** | Atomic factual statements (NLI-extracted) | Fact-heavy corpora |

### Late Chunking

Traditional chunking embeds each chunk in isolation context is lost. **Late chunking** runs the full document through a long-context encoder (e.g., `jina-embeddings-v2-base-en` with 8 K context), then segments the token-level embeddings after the fact. Each chunk embedding contains the full document's contextual signal.

### Hierarchical (Parent-Child) Chunking

- **Index** child chunks (small, ~128 tokens) for precise retrieval
- **Return** the parent chunk (large, ~512 tokens) to the LLM for context
- Net effect: precision of small chunks, context richness of large chunks

In [2]:
# pip install langchain langchain-community

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

SAMPLE_TEXT = """
Retrieval-Augmented Generation (RAG) combines parametric memory (LLM weights)
with non-parametric memory (a document store). At inference time the model
retrieves relevant passages and conditions its generation on them.

The original RAG paper by Lewis et al. (2020) showed that retrieval helps
on knowledge-intensive tasks like Natural Questions and TriviaQA. Later work
extended this to multi-hop reasoning, long-form generation, and dialogue.

## Chunking

### Fixed-Size
The simplest approach: split every N tokens with M tokens of overlap.

### Semantic
Detect topic shifts by monitoring cosine similarity between consecutive sentences.
"""

# ── 1. Fixed-size ─────────────────────────────────────────────────────────────
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    length_function=len,
)
fixed_chunks = fixed_splitter.split_text(SAMPLE_TEXT)
print(f"Fixed-size: {len(fixed_chunks)} chunks")
for i, c in enumerate(fixed_chunks):
    print(f"  [{i}] ({len(c)} chars): {c[:80].strip()!r}")

# ── 2. Recursive character splitting ─────────────────────────────────────────
# Tries \n\n first, then \n, then " ", then ""
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=250,
    chunk_overlap=30,
)
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)
print(f"\nRecursive: {len(recursive_chunks)} chunks")

# ── 3. Markdown header splitting ──────────────────────────────────────────────
headers_to_split_on = [
    ("#",  "H1"),
    ("##", "H2"),
    ("###","H3"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks   = md_splitter.split_text(SAMPLE_TEXT)
print(f"\nMarkdown-header: {len(md_chunks)} chunks")
for c in md_chunks:
    print(f"  metadata={c.metadata}  content={c.page_content[:60]!r}")

Fixed-size: 5 chunks
  [0] (152 chars): 'Retrieval-Augmented Generation (RAG) combines parametric memory (LLM weights)\nwi'
  [1] (66 chars): 'retrieves relevant passages and conditions its generation on them.'
  [2] (150 chars): 'The original RAG paper by Lewis et al. (2020) showed that retrieval helps\non kno'
  [3] (73 chars): 'extended this to multi-hop reasoning, long-form generation, and dialogue.'
  [4] (194 chars): '## Chunking\n\n### Fixed-Size\nThe simplest approach: split every N tokens with M t'

Recursive: 3 chunks

Markdown-header: 3 chunks
  metadata={}  content='Retrieval-Augmented Generation (RAG) combines parametric mem'
  metadata={'H2': 'Chunking', 'H3': 'Fixed-Size'}  content='The simplest approach: split every N tokens with M tokens of'
  metadata={'H2': 'Chunking', 'H3': 'Semantic'}  content='Detect topic shifts by monitoring cosine similarity between '


In [3]:
# Semantic chunking split where cosine similarity between consecutive sentences drops
import numpy as np
from typing import List

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def semantic_chunking(
    sentences: List[str],
    embed_fn,          # callable: List[str] -> np.ndarray of shape (N, D)
    threshold: float = 0.75,
) -> List[List[str]]:
    """
    Split a document into chunks where cosine similarity between
    consecutive sentence embeddings drops below `threshold`.
    """
    if not sentences:
        return []

    embeddings = embed_fn(sentences)   # shape: (N, D)
    chunks: List[List[str]] = []
    current: List[str] = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(embeddings[i - 1], embeddings[i])
        if sim < threshold:
            chunks.append(current)     # topic shift detected
            current = []
        current.append(sentences[i])

    if current:
        chunks.append(current)

    return chunks

# Demo with mock embeddings
def mock_embed(sentences):
    """Mock: first 3 sentences share one direction, last 2 another."""
    rng = np.random.default_rng(42)
    base_a = np.array([1.0, 0.0, 0.0])
    base_b = np.array([0.0, 1.0, 0.0])
    embs = []
    for i, _ in enumerate(sentences):
        base = base_a if i < 3 else base_b
        noise = rng.normal(0, 0.05, size=3)
        v = base + noise
        embs.append(v / np.linalg.norm(v))
    return np.array(embs)

sentences = [
    "RAG combines retrieval with generation.",
    "The retriever fetches relevant documents.",
    "The generator conditions on those documents.",
    "ColBERT uses late interaction for reranking.",
    "MaxSim aggregates token-level scores.",
]
chunks = semantic_chunking(sentences, mock_embed, threshold=0.5)
print("Semantic chunks:")
for i, ch in enumerate(chunks):
    print(f"  Chunk {i}: {ch}")

Semantic chunks:
  Chunk 0: ['RAG combines retrieval with generation.', 'The retriever fetches relevant documents.', 'The generator conditions on those documents.']
  Chunk 1: ['ColBERT uses late interaction for reranking.', 'MaxSim aggregates token-level scores.']


In [4]:
# Hierarchical (Parent-Child) chunking with LangChain

from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, List
import uuid

def hierarchical_chunk(
    text: str,
    parent_chunk_size: int = 512,
    child_chunk_size:  int = 128,
    overlap: int = 20,
) -> List[Dict]:
    """
    Returns a list of dicts, each with:
      - parent_id   : UUID for the parent chunk
      - parent_text : full parent context given to the LLM
      - child_id    : UUID for the child chunk (what gets indexed)
      - child_text  : small chunk used for retrieval
    """
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=parent_chunk_size, chunk_overlap=overlap)
    child_splitter  = RecursiveCharacterTextSplitter(
        chunk_size=child_chunk_size,  chunk_overlap=overlap)

    records = []
    for parent_text in parent_splitter.split_text(text):
        parent_id = str(uuid.uuid4())
        for child_text in child_splitter.split_text(parent_text):
            records.append({
                "parent_id":   parent_id,
                "parent_text": parent_text,
                "child_id":    str(uuid.uuid4()),
                "child_text":  child_text,
            })
    return records

long_text = SAMPLE_TEXT * 4   # repeat to get meaningful parents
records = hierarchical_chunk(long_text)
print(f"Total parent-child pairs: {len(records)}")
print("\nSample record:")
r = records[0]
print(f"  parent_id : {r['parent_id']}")
print(f"  child_text: {r['child_text'][:80]!r}")
print(f"  parent_text (first 120 chars): {r['parent_text'][:120]!r}")

Total parent-child pairs: 34

Sample record:
  parent_id : deecce0e-9a27-4cb5-a574-5bd004110b75
  child_text: 'Retrieval-Augmented Generation (RAG) combines parametric memory (LLM weights)'
  parent_text (first 120 chars): 'Retrieval-Augmented Generation (RAG) combines parametric memory (LLM weights)\nwith non-parametric memory (a document sto'


## 3. Query Transformation Techniques

Even with a perfect index, a poorly-phrased query will miss relevant documents. Query transformation fixes the *query side* of the recall problem.

### Techniques Overview

| Technique | Idea | When to Use |
|---|---|---|
| **Query rewriting** | LLM rewrites the query for clarity/specificity | Noisy or ambiguous user input |
| **HyDE** | Generate a *hypothetical* answer, embed that | Query-document vocabulary mismatch |
| **Multi-query** | Generate N paraphrases, retrieve for all, deduplicate | Low recall with single query |
| **Step-back** | Abstract to a more general question first | Highly specific queries |
| **RAG-Fusion** | Multi-query + Reciprocal Rank Fusion | Best overall recall |

### HyDE Hypothetical Document Embedding

The intuition: a *document* that answers the question lives in a different part of the embedding space than the *question itself*. By generating a fake answer first, we move the query vector into the document space.

$$\hat{d} = \text{LLM}(q)$$

Then retrieve using $\text{sim}(\hat{d},\, d_i)$ instead of $\text{sim}(q,\, d_i)$.

### RAG-Fusion with Reciprocal Rank Fusion

Generate $|Q|$ query variants. For each variant retrieve a ranked list. Combine with RRF:

$$\text{RRF}(d) = \sum_{q \in Q} \frac{1}{k + \text{rank}_q(d)}, \quad k = 60$$

The constant $k=60$ was chosen empirically it dampens the effect of very high ranks while preserving the contribution of lower-ranked documents. Results are then sorted by $\text{RRF}(d)$ descending.

In [5]:
# Query Transformation Implementations
# Requires: pip install langchain-openai langchain

from typing import List, Dict, Tuple
from dataclasses import dataclass

@dataclass
class MockDoc:
    page_content: str
    metadata: Dict

# ── HyDE ──────────────────────────────────────────────────────────────────────
def hyde_query(question: str, llm, retriever):
    """
    Hypothetical Document Embedding:
      1. Ask the LLM to write a passage that would answer the question.
      2. Embed that hypothetical passage.
      3. Use it as the retrieval query.
    """
    hyde_prompt = (
        f"Write a detailed passage (3-5 sentences) that directly answers the "
        f"following question. Do not mention that you are answering a question.\n\n"
        f"Question: {question}"
    )
    hypothetical_doc = llm.invoke(hyde_prompt)          # LangChain message
    content = getattr(hypothetical_doc, "content", str(hypothetical_doc))
    results = retriever.invoke(content)
    return results, content


# ── Multi-query generation ────────────────────────────────────────────────────
def generate_multi_queries(question: str, llm, n: int = 3) -> List[str]:
    """
    Ask the LLM to rephrase the question N ways to improve recall.
    """
    prompt = (
        f"Generate {n} different phrasings of the following question. "
        f"Each phrasing should capture the same intent but use different vocabulary.\n"
        f"Return ONLY the questions, one per line.\n\n"
        f"Question: {question}"
    )
    response = llm.invoke(prompt)
    content  = getattr(response, "content", str(response))
    queries  = [q.strip() for q in content.strip().split("\n") if q.strip()]
    return [question] + queries[:n]   # include original


# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def reciprocal_rank_fusion(
    results_list: List[List[MockDoc]],
    k: int = 60,
) -> List[Tuple[str, float]]:
    """
    RRF(d) = sum over queries of 1 / (k + rank(d))
    Returns list of (doc_id, score) sorted descending.
    """
    fused_scores: Dict[str, float] = {}
    doc_map:      Dict[str, MockDoc] = {}

    for results in results_list:
        for rank, doc in enumerate(results):
            doc_id = doc.metadata.get("id", doc.page_content[:80])
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
            doc_map[doc_id] = doc

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked


# ── RAG-Fusion pipeline ───────────────────────────────────────────────────────
def rag_fusion(question: str, llm, retriever, n_queries: int = 3):
    queries  = generate_multi_queries(question, llm, n=n_queries)
    all_results = [retriever.invoke(q) for q in queries]
    ranked   = reciprocal_rank_fusion(all_results)
    return queries, ranked


# ── Demo with mock objects ────────────────────────────────────────────────────
mock_docs = [
    MockDoc("RAG uses retrieval to augment generation.",  {"id": "doc1"}),
    MockDoc("ColBERT performs late interaction reranking.",{"id": "doc2"}),
    MockDoc("HyDE generates hypothetical documents.",     {"id": "doc3"}),
    MockDoc("BM25 is a sparse retrieval algorithm.",      {"id": "doc4"}),
]

# Simulate three query result sets with different orderings
results_q1 = mock_docs[:3]
results_q2 = [mock_docs[1], mock_docs[3], mock_docs[0]]
results_q3 = [mock_docs[2], mock_docs[0], mock_docs[3]]

rrf_scores = reciprocal_rank_fusion([results_q1, results_q2, results_q3])
print("RRF-fused ranking:")
for doc_id, score in rrf_scores:
    print(f"  {doc_id}  score={score:.4f}")

RRF-fused ranking:
  doc1  score=0.0484
  doc2  score=0.0325
  doc3  score=0.0323
  doc4  score=0.0320


In [6]:
# Step-back prompting demo

def step_back_query(question: str, llm) -> str:
    """
    Generate a more general, abstract version of the question.
    Then retrieve on both the original AND the step-back question.
    """
    prompt = (
        "You are a helpful assistant. Given a specific question, generate a more general "
        "question that captures the underlying concept needed to answer it.\n\n"
        f"Specific question: {question}\n"
        "General question:"
    )
    response = llm.invoke(prompt)
    return getattr(response, "content", str(response)).strip()


# Example without running LLM
examples = [
    {
        "original":  "What is the boiling point of ethanol at 2 atm pressure?",
        "step_back": "What is the relationship between pressure and boiling point?",
    },
    {
        "original":  "Who won the 1987 NBA championship?",
        "step_back": "What is the history of NBA championships?",
    },
    {
        "original":  "How do I configure HNSW ef parameter in Milvus?",
        "step_back": "How does HNSW indexing work in vector databases?",
    },
]

print("Step-back prompting examples:")
for ex in examples:
    print(f"  Original:  {ex['original']}")
    print(f"  Step-back: {ex['step_back']}")
    print()

Step-back prompting examples:
  Original:  What is the boiling point of ethanol at 2 atm pressure?
  Step-back: What is the relationship between pressure and boiling point?

  Original:  Who won the 1987 NBA championship?
  Step-back: What is the history of NBA championships?

  Original:  How do I configure HNSW ef parameter in Milvus?
  Step-back: How does HNSW indexing work in vector databases?



## 4. Reranking

### Why First-Stage Retrieval Is Imprecise

ANN indexes trade accuracy for speed. They return the *approximately* nearest neighbours some relevant documents are missed, and some irrelevant documents sneak in. The tradeoff:

- HNSW with `ef=64`: fast, ~95% recall@10
- HNSW with `ef=512`: slower, ~99.9% recall@10

Rather than paying the cost of high-`ef` ANN for every query, the industry pattern is:

1. **First stage**: fast ANN, retrieve top-k=50-200 candidates
2. **Second stage**: accurate reranker, score each candidate, return top-5

### Cross-Encoders

A bi-encoder (standard RAG) encodes query and document *separately*, then computes cosine similarity. A **cross-encoder** takes `[CLS] query [SEP] document [SEP]` as a single input and outputs a relevance score it can model *interactions* between query and document tokens.

| Model | Encoding | Complexity | Accuracy |
|---|---|---|---|
| Bi-encoder | Separate | $O(1)$ at query time | Medium |
| Cross-encoder | Joint | $O(n)$ per query | High |
| ColBERT | Late interaction | $O(|q| \cdot |d|)$ MaxSim | High |

### ColBERT / ColBERTv2 Late Interaction

ColBERT encodes each query token and each document token into its own vector. The score is computed via MaxSim:

$$S(q,\, d) = \sum_{i \in q} \max_{j \in d}\; E_{q_i} \cdot E_{d_j}^T$$

Each query token "finds" its best matching document token. This allows fine-grained token-level alignment while keeping document embeddings pre-computable.

### Reranking Taxonomy

| Paradigm | Method | Description |
|---|---|---|
| **Pointwise** | Cross-encoder, BGE, Cohere | Score each (query, doc) pair independently |
| **Pairwise** | DuoT5, RankNet | Compare pairs of documents |
| **Listwise** | RankGPT, LLM-as-ranker | Score the entire ranked list at once |

### Popular Rerankers

| Model | Notes |
|---|---|
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | Fast, good quality, open source |
| `BAAI/bge-reranker-large` | Strong multilingual reranker |
| Cohere `rerank-english-v3.0` | API-based, very strong |
| RAGatouille (ColBERT) | 3-line ColBERT interface |
| FlashRank | Lightweight, CPU-friendly |
| Jina Reranker | `jina-reranker-v2-base-multilingual` |
| MonoT5 | T5-based pointwise reranker |
| RankGPT | GPT-4 as listwise ranker |

In [7]:
# Reranking pipeline implementations
# pip install sentence-transformers FlagEmbedding cohere ragatouille

from typing import List, Tuple
import numpy as np

# ── Cross-encoder (sentence-transformers) ─────────────────────────────────────
def cross_encoder_rerank(
    query: str,
    candidates: List[str],
    model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k: int = 5,
) -> List[Tuple[int, float, str]]:
    """
    Re-rank candidates using a cross-encoder.
    Returns list of (original_rank, score, text) sorted by score desc.
    """
    from sentence_transformers import CrossEncoder
    model  = CrossEncoder(model_name)
    pairs  = [[query, c] for c in candidates]
    scores = model.predict(pairs)
    ranked = sorted(
        [(i, float(s), candidates[i]) for i, s in enumerate(scores)],
        key=lambda x: x[1], reverse=True
    )
    return ranked[:top_k]


# ── BGE reranker (FlagEmbedding) ──────────────────────────────────────────────
def bge_rerank(
    query: str,
    candidates: List[str],
    model_name: str = "BAAI/bge-reranker-large",
    top_k: int = 5,
) -> List[Tuple[int, float, str]]:
    from FlagEmbedding import FlagReranker
    reranker = FlagReranker(model_name, use_fp16=True)
    pairs    = [[query, c] for c in candidates]
    scores   = reranker.compute_score(pairs)
    ranked = sorted(
        [(i, float(s), candidates[i]) for i, s in enumerate(scores)],
        key=lambda x: x[1], reverse=True
    )
    return ranked[:top_k]


# ── Cohere Rerank API ──────────────────────────────────────────────────────────
def cohere_rerank(
    query: str,
    candidates: List[str],
    api_key: str,
    model: str = "rerank-english-v3.0",
    top_k: int = 5,
):
    import cohere
    co      = cohere.Client(api_key)
    results = co.rerank(
        query=query,
        documents=candidates,
        model=model,
        top_n=top_k,
    )
    return [(r.index, r.relevance_score, candidates[r.index]) for r in results.results]


# ── RAGatouille (ColBERT) ─────────────────────────────────────────────────────
RAGATOUILLE_EXAMPLE = '''
from ragatouille import RAGPretrainedModel

RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

# Index documents
RAG.index(collection=documents, index_name="my_index")

# Retrieve
results = RAG.search(query="What is late interaction?", k=5)
# Each result: {"content": ..., "score": ..., "rank": ...}
'''
print("RAGatouille (ColBERT) usage:")
print(RAGATOUILLE_EXAMPLE)


# ── Mock demo ─────────────────────────────────────────────────────────────────
def mock_rerank(query: str, candidates: List[str], top_k: int = 3):
    """Simple length-based mock reranker for demo purposes."""
    # In reality replace with a real cross-encoder
    scores = [len(set(query.lower().split()) & set(c.lower().split())) for c in candidates]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(i, s, candidates[i]) for i, s in ranked[:top_k]]

query = "how does retrieval augmented generation work"
candidates = [
    "RAG combines retrieval with neural generation.",
    "The recipe calls for flour, eggs, and butter.",
    "Retrieval augmented generation uses a document store to augment LLM generation.",
    "HNSW is an index algorithm for approximate nearest neighbours.",
    "Retrieval is the process of fetching relevant documents for a query.",
]
print("\nMock reranking results:")
for rank, (orig_idx, score, text) in enumerate(mock_rerank(query, candidates)):
    print(f"  #{rank+1} (orig pos {orig_idx}, score={score}): {text[:70]}")

RAGatouille (ColBERT) usage:

from ragatouille import RAGPretrainedModel

RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

# Index documents
RAG.index(collection=documents, index_name="my_index")

# Retrieve
results = RAG.search(query="What is late interaction?", k=5)
# Each result: {"content": ..., "score": ..., "rank": ...}


Mock reranking results:
  #1 (orig pos 2, score=3): Retrieval augmented generation uses a document store to augment LLM ge
  #2 (orig pos 0, score=1): RAG combines retrieval with neural generation.
  #3 (orig pos 4, score=1): Retrieval is the process of fetching relevant documents for a query.


## 5. Advanced RAG Architectures

Beyond retrieve-then-generate, several architectures improve quality by changing *when*, *what*, and *whether* to retrieve.

### RAPTOR Recursive Abstractive Processing for Tree-Organized Retrieval

1. Chunk documents into leaf nodes
2. Cluster leaf nodes (Gaussian Mixture Models / k-means)
3. Summarize each cluster → new parent nodes
4. Repeat until a single root node
5. Index **all** nodes (leaves + internal)
6. Retrieve at any level depending on query abstraction

Net effect: handles both fine-grained lookup (leaves) and broad synthesis (root) in the same index.

### LightRAG

Graph + vector hybrid. Extracts entities and relations from documents using an LLM, builds a knowledge graph, then supports two retrieval modes:
- **Local**: entity-centric, finds context around specific entities
- **Global**: graph-traversal for cross-entity reasoning

### HippoRAG Hippocampus-Inspired RAG

Inspired by the human hippocampus memory system (neocortex ↔ hippocampus synergy). Extracts a knowledge graph, then uses **Personalized PageRank (PPR)** to propagate retrieval signal through the graph:

$$\text{PPR}(v) = (1 - \alpha)\,\mathbf{M}\,\text{PPR}(v) + \alpha\,\mathbf{s}$$

where $\mathbf{s}$ is the personalisation vector (seeded at query-relevant nodes) and $\alpha$ is the restart probability.

### Self-RAG

A single model trained to generate **reflection tokens** that control retrieval and assess output quality:

| Token | Meaning |
|---|---|
| `[Retrieve]` | Should I retrieve? |
| `[IsRel]` | Is this retrieved passage relevant? |
| `[IsSup]` | Is my generation supported by the passage? |
| `[IsUse]` | Is my response useful to the user? |

### CRAG Corrective RAG

Adds a **retrieval evaluator** that scores retrieved documents. If confidence is low → trigger a web search fallback (e.g., Tavily Search). Combines local retrieval with live web data.

### FLARE Forward-Looking Active Retrieval

Only retrieves when the model has low confidence on upcoming tokens. Algorithm:
1. Generate tokens greedily
2. If any token probability < threshold $\theta$, mask those tokens and retrieve
3. Continue generation conditioned on retrieved context

### Speculative RAG

A small **drafter** model generates a RAG answer quickly; a large **verifier** model checks it. Inspired by speculative decoding.

In [8]:
# Self-RAG style conditional retrieval
from typing import Optional, Tuple

def self_rag_decide_retrieve(
    query: str,
    llm,
    threshold: float = 0.7,
) -> Tuple[bool, float, str]:
    """
    Ask the LLM whether retrieval is needed.
    Returns (should_retrieve, confidence, reasoning).
    """
    prompt = (
        f"Question: {query}\n\n"
        "Can you answer this question accurately from your training knowledge alone?\n"
        "Respond with exactly this format:\n"
        "ANSWER: YES or NO\n"
        "CONFIDENCE: 0.0-1.0\n"
        "REASON: one sentence"
    )
    response = llm.invoke(prompt)
    text     = getattr(response, "content", str(response))

    # Parse response
    lines = {}
    for line in text.strip().split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            lines[k.strip().upper()] = v.strip()

    can_answer  = lines.get("ANSWER", "NO").upper() == "YES"
    confidence  = float(lines.get("CONFIDENCE", "0.5"))
    reason      = lines.get("REASON", "")
    should_retrieve = (not can_answer) or (confidence < threshold)

    return should_retrieve, confidence, reason


def self_rag_pipeline(
    query: str,
    llm,
    retriever,
    threshold: float = 0.7,
) -> dict:
    """
    Full Self-RAG style pipeline:
      1. Decide whether to retrieve
      2. If yes: retrieve → assess relevance → generate → assess support
      3. If no: generate directly from parametric knowledge
    """
    should_retrieve, confidence, reason = self_rag_decide_retrieve(query, llm, threshold)

    if not should_retrieve:
        answer = llm.invoke(query)
        return {
            "retrieved": False,
            "confidence": confidence,
            "reason": reason,
            "answer": getattr(answer, "content", str(answer)),
            "contexts": [],
        }

    # Retrieve
    contexts = retriever.invoke(query)

    # Assess relevance for each context  [IsRel]
    relevant_contexts = []
    for ctx in contexts:
        rel_prompt = (
            f"Question: {query}\n"
            f"Passage: {ctx.page_content}\n\n"
            "Is this passage relevant to answering the question? Answer YES or NO."
        )
        rel_response = llm.invoke(rel_prompt)
        if "YES" in getattr(rel_response, "content", str(rel_response)).upper():
            relevant_contexts.append(ctx)

    # Generate with relevant context
    context_str = "\n\n".join(c.page_content for c in relevant_contexts)
    gen_prompt  = f"Context:\n{context_str}\n\nQuestion: {query}\nAnswer:"
    answer      = llm.invoke(gen_prompt)

    return {
        "retrieved":  True,
        "confidence": confidence,
        "reason":     reason,
        "answer":     getattr(answer, "content", str(answer)),
        "contexts":   [c.page_content for c in relevant_contexts],
    }


# Demo structure (requires an actual LLM + retriever to run)
print("Self-RAG pipeline flow:")
print("  1. [Retrieve?] → assess whether parametric knowledge suffices")
print("  2. [IsRel]     → filter irrelevant retrieved passages")
print("  3. Generate    → condition on relevant context")
print("  4. [IsSup]     → verify generation is grounded in context")
print("  5. [IsUse]     → verify response is useful")

Self-RAG pipeline flow:
  1. [Retrieve?] → assess whether parametric knowledge suffices
  2. [IsRel]     → filter irrelevant retrieved passages
  3. Generate    → condition on relevant context
  4. [IsSup]     → verify generation is grounded in context
  5. [IsUse]     → verify response is useful


In [9]:
# CRAG Corrective RAG: retrieval quality evaluator with web fallback

from enum import Enum
from typing import List

class RetrievalQuality(Enum):
    CORRECT   = "correct"      # ≥ 1 highly relevant doc found
    AMBIGUOUS = "ambiguous"    # some relevant, some not
    INCORRECT = "incorrect"    # no relevant docs found


def evaluate_retrieval_quality(
    query: str,
    contexts: List[str],
    llm,
    correct_threshold: float = 0.7,
    incorrect_threshold: float = 0.3,
) -> RetrievalQuality:
    """
    Score each retrieved document for relevance.
    Aggregate into CORRECT / AMBIGUOUS / INCORRECT.
    """
    scores = []
    for ctx in contexts:
        prompt = (
            f"Rate the relevance of this passage to the query on a scale 0-1.\n"
            f"Query: {query}\nPassage: {ctx}\n"
            "Output ONLY a number between 0 and 1."
        )
        resp = llm.invoke(prompt)
        try:
            score = float(getattr(resp, "content", str(resp)).strip())
        except ValueError:
            score = 0.5
        scores.append(score)

    if not scores:
        return RetrievalQuality.INCORRECT

    max_score = max(scores)
    if max_score >= correct_threshold:
        return RetrievalQuality.CORRECT
    elif max_score <= incorrect_threshold:
        return RetrievalQuality.INCORRECT
    else:
        return RetrievalQuality.AMBIGUOUS


def crag_pipeline(query: str, llm, retriever, web_search_fn) -> dict:
    """
    CRAG: local retrieval → evaluate quality → web fallback if needed.
    """
    # First: local retrieval
    local_results  = retriever.invoke(query)
    local_texts    = [r.page_content for r in local_results]

    quality = evaluate_retrieval_quality(query, local_texts, llm)

    if quality == RetrievalQuality.CORRECT:
        final_contexts = local_texts
        source = "local"
    elif quality == RetrievalQuality.INCORRECT:
        # Rewrite query for web search and retrieve from web
        web_results    = web_search_fn(query)
        final_contexts = web_results
        source = "web"
    else:  # AMBIGUOUS combine both
        web_results    = web_search_fn(query)
        final_contexts = local_texts + web_results
        source = "hybrid"

    context_str = "\n\n".join(final_contexts)
    answer_prompt = f"Context:\n{context_str}\n\nQuestion: {query}\nAnswer:"
    answer = llm.invoke(answer_prompt)

    return {
        "quality": quality.value,
        "source":  source,
        "answer":  getattr(answer, "content", str(answer)),
    }


print("CRAG decision logic:")
print("  CORRECT   → use local retrieval directly")
print("  AMBIGUOUS → combine local + web")
print("  INCORRECT → fall back to web search entirely")

CRAG decision logic:
  CORRECT   → use local retrieval directly
  AMBIGUOUS → combine local + web
  INCORRECT → fall back to web search entirely


## 6. Hybrid Search

No single retrieval method dominates across all query types.

| Method | Strength | Weakness |
|---|---|---|
| **Dense (vector)** | Handles synonyms, paraphrases, semantic similarity | Fails on rare terms, product codes, exact strings |
| **Sparse (BM25)** | Exact keyword matching, rare term recall | No semantic understanding |

Hybrid search combines both signals.

### Linear Combination

Normalise scores to [0, 1] then combine:

$$s_{hybrid}(d) = \alpha \cdot s_{dense}(d) + (1 - \alpha) \cdot s_{sparse}(d)$$

Problem: score scales differ between methods; calibrating $\alpha$ is fragile.

### Reciprocal Rank Fusion (preferred)

RRF operates on *ranks* not scores, so no calibration needed:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + r(d)}, \quad k = 60$$

### SPLADE Sparse Learned Efficient Retrieval and Domain Adaptation

SPLADE produces *learned sparse* representations by running a BERT-based model and projecting to the full vocabulary (30 K tokens). Each document gets a sparse vector with non-zero weights for semantically related terms bridging dense and sparse worlds.

The weight for vocabulary token $j$ is:

$$w_j = \log\!\left(1 + \text{ReLU}(h_j)\right)$$

where $h_j$ is the logit from the MLM head. The $\log(1+\cdot)$ bounds weights and the $\text{ReLU}$ ensures non-negativity. This makes SPLADE interpretable you can see *which terms* a document was expanded to, unlike a dense vector.

In [10]:
# Hybrid search: BM25 + dense vector + RRF
# pip install rank-bm25 numpy

import numpy as np
from typing import List, Dict, Tuple

try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except ImportError:
    BM25_AVAILABLE = False
    print("rank-bm25 not installed using mock BM25")


class MockBM25:
    """Minimal BM25 mock for demo without rank_bm25 installed."""
    def __init__(self, corpus: List[List[str]]):
        self.corpus = corpus

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        qt = set(query_tokens)
        return np.array([len(qt & set(doc)) for doc in self.corpus], dtype=float)


def build_bm25(texts: List[str]):
    tokenised = [t.lower().split() for t in texts]
    if BM25_AVAILABLE:
        return BM25Okapi(tokenised)
    return MockBM25(tokenised)


def rrf_fuse(
    rankings: List[List[int]],   # each list = ranked doc indices from one retriever
    k: int = 60,
) -> List[Tuple[int, float]]:
    """RRF over multiple ranked lists of document indices."""
    scores: Dict[int, float] = {}
    for ranked_list in rankings:
        for rank, doc_idx in enumerate(ranked_list):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(
    query: str,
    documents: List[str],
    doc_embeddings: np.ndarray,   # shape (N, D)
    query_embedding: np.ndarray,  # shape (D,)
    top_k: int = 5,
    k_rrf: int = 60,
) -> List[Tuple[int, float, str]]:
    """
    Hybrid retrieval combining BM25 and dense cosine similarity.
    Returns top_k documents by RRF score.
    """
    n = len(documents)

    # ── BM25 ──────────────────────────────────────────────────────────────────
    bm25        = build_bm25(documents)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranked = list(np.argsort(bm25_scores)[::-1])

    # ── Dense cosine ──────────────────────────────────────────────────────────
    norms  = np.linalg.norm(doc_embeddings, axis=1, keepdims=True) + 1e-9
    normed = doc_embeddings / norms
    qnorm  = query_embedding / (np.linalg.norm(query_embedding) + 1e-9)
    cos_scores = normed @ qnorm
    dense_ranked = list(np.argsort(cos_scores)[::-1])

    # ── RRF fusion ────────────────────────────────────────────────────────────
    fused = rrf_fuse([bm25_ranked, dense_ranked], k=k_rrf)

    results = [(idx, score, documents[idx]) for idx, score in fused[:top_k]]
    return results


# ── Demo ──────────────────────────────────────────────────────────────────────
np.random.seed(42)
documents = [
    "Retrieval augmented generation uses a document store.",
    "BM25 is a classic sparse retrieval algorithm.",
    "HNSW provides approximate nearest neighbour search.",
    "ColBERT uses late interaction for reranking.",
    "Dense retrieval encodes queries and documents into vectors.",
    "Hybrid search combines sparse and dense retrieval.",
    "SPLADE produces learned sparse representations.",
]

D = 8   # toy embedding dimension
doc_embeddings   = np.random.randn(len(documents), D)
query_embedding  = np.random.randn(D)

# Bias query embedding toward "retrieval" documents (indices 0, 1, 4, 5)
for i in [0, 1, 4, 5]:
    doc_embeddings[i] += query_embedding * 0.5

query   = "dense and sparse retrieval combination"
results = hybrid_search(query, documents, doc_embeddings, query_embedding, top_k=5)

print(f"Query: '{query}'\n")
print("Hybrid search results (RRF):")
for rank, (idx, score, text) in enumerate(results):
    print(f"  #{rank+1} (doc {idx}, RRF={score:.4f}): {text}")

Query: 'dense and sparse retrieval combination'

Hybrid search results (RRF):
  #1 (doc 5, RRF=0.0323): Hybrid search combines sparse and dense retrieval.
  #2 (doc 1, RRF=0.0323): BM25 is a classic sparse retrieval algorithm.
  #3 (doc 4, RRF=0.0323): Dense retrieval encodes queries and documents into vectors.
  #4 (doc 0, RRF=0.0310): Retrieval augmented generation uses a document store.
  #5 (doc 6, RRF=0.0306): SPLADE produces learned sparse representations.


In [11]:
# SPLADE weight visualisation (conceptual shows the interpretability)
import numpy as np

def splade_weights_mock(text: str, vocab_subset: List[str]) -> Dict[str, float]:
    """
    Mock SPLADE: shows log(1 + ReLU(h)) weighting.
    In production use: https://huggingface.co/naver/splade-cocondenser-ensembledistil
    """
    rng = np.random.default_rng(hash(text) % (2**32))
    # Tokens present in text get higher raw logits
    text_tokens = set(text.lower().split())
    weights = {}
    for token in vocab_subset:
        raw_logit = rng.normal(0.5 if token in text_tokens else -0.5, 0.3)
        w = np.log(1 + max(0.0, raw_logit))   # log(1 + ReLU(h))
        if w > 0.01:
            weights[token] = round(float(w), 4)
    return dict(sorted(weights.items(), key=lambda x: x[1], reverse=True))


vocab = ["retrieval", "augmented", "generation", "search", "document",
         "query", "embedding", "sparse", "dense", "vector", "index",
         "rank", "score", "semantic", "keyword", "knowledge"]

text  = "retrieval augmented generation uses document embeddings"
weights = splade_weights_mock(text, vocab)
print(f"SPLADE weights for: '{text}'")
print("Token weights (log(1+ReLU(h))):")
for token, w in list(weights.items())[:8]:
    bar = "█" * int(w * 30)
    print(f"  {token:<15} {w:.4f}  {bar}")
print("\nInterpretability: you can see which terms the document was expanded to.")

SPLADE weights for: 'retrieval augmented generation uses document embeddings'
Token weights (log(1+ReLU(h))):
  retrieval       0.6187  ██████████████████
  document        0.4310  ████████████
  augmented       0.3664  ██████████
  generation      0.3095  █████████
  sparse          0.1756  █████
  keyword         0.0994  ██

Interpretability: you can see which terms the document was expanded to.


## 7. RAG Evaluation

A RAG pipeline has multiple failure modes evaluation must cover all of them.

### Failure Mode Taxonomy

| Failure | Description | Metric |
|---|---|---|
| Wrong chunks retrieved | Retriever misses relevant docs | Context Recall |
| Irrelevant chunks included | Retriever returns noise | Context Precision |
| Hallucination | Answer contains claims not in context | Faithfulness |
| Off-topic answer | Answer doesn't address the question | Answer Relevancy |

### RAGAS Metrics

| Metric | What it measures | Range |
|---|---|---|
| **Answer Relevancy** | Is the answer relevant to the question? | [0, 1] |
| **Faithfulness** | Is the answer grounded in the retrieved context? | [0, 1] |
| **Context Precision** | Are the retrieved chunks actually relevant? | [0, 1] |
| **Context Recall** | Does the retrieved set cover the ground truth? | [0, 1] |

RAGAS uses LLM-as-judge internally GPT-4 or a specified judge model scores each dimension.

### Other Evaluation Frameworks

- **TruLens**: wraps any LLM app with feedback functions; records traces in a dashboard
- **DeepEval**: `pip install deepeval`; G-Eval, hallucination, toxicity, contextual relevancy
- **BEIR benchmark**: 18 heterogeneous retrieval datasets for zero-shot evaluation
- **MTEB**: Massive Text Embedding Benchmark standard for embedding model evaluation

### Evaluation Dataset Construction

```
question      → what the user asked
answer        → what the RAG pipeline returned
contexts      → list of retrieved passages used
ground_truth  → the ideal answer (for recall computation)
```

In [12]:
# RAGAS evaluation pipeline
# pip install ragas datasets

RAGAS_EXAMPLE = '''
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)
from datasets import Dataset

# Build evaluation dataset
eval_data = {
    "question": [
        "What is RAG?",
        "How does ColBERT work?",
        "What is HNSW?",
    ],
    "answer": [
        "RAG combines retrieval with generation to augment LLM responses.",
        "ColBERT uses late interaction with per-token embeddings and MaxSim.",
        "HNSW is a graph-based ANN index with O(log n) search complexity.",
    ],
    "contexts": [
        ["Retrieval-Augmented Generation (RAG) fetches relevant documents at inference time."],
        ["ColBERT encodes each token independently and uses MaxSim for scoring."],
        ["HNSW builds a hierarchical graph for approximate nearest neighbour search."],
    ],
    "ground_truth": [
        "RAG is a technique that retrieves relevant documents and conditions LLM generation on them.",
        "ColBERT is a late interaction model using per-token MaxSim aggregation.",
        "HNSW (Hierarchical Navigable Small World) is an ANN algorithm with log-n complexity.",
    ],
}

dataset = Dataset.from_dict(eval_data)

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
)

print(result)
# Output: {'faithfulness': 0.93, 'answer_relevancy': 0.91,
#           'context_precision': 0.88, 'context_recall': 0.85}

df = result.to_pandas()
print(df[["question", "faithfulness", "answer_relevancy"]])
'''
print("RAGAS evaluation pipeline:")
print(RAGAS_EXAMPLE)

RAGAS evaluation pipeline:



from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)
from datasets import Dataset

# Build evaluation dataset
eval_data = {
    "question": [
        "What is RAG?",
        "How does ColBERT work?",
        "What is HNSW?",
    ],
    "answer": [
        "RAG combines retrieval with generation to augment LLM responses.",
        "ColBERT uses late interaction with per-token embeddings and MaxSim.",
        "HNSW is a graph-based ANN index with O(log n) search complexity.",
    ],
    "contexts": [
        ["Retrieval-Augmented Generation (RAG) fetches relevant documents at inference time."],
        ["ColBERT encodes each token independently and uses MaxSim for scoring."],
        ["HNSW builds a hierarchical graph for approximate nearest neighbour search."],
    ],
    "ground_truth": [
        "RAG is a technique that retrieves relevant documents and conditions LLM generation on them.",
        

In [13]:
# Manual faithfulness evaluator (works without RAGAS)
from typing import List, Tuple

def manual_faithfulness_score(
    answer: str,
    contexts: List[str],
    llm,
) -> Tuple[float, List[str]]:
    """
    Split answer into claims, then verify each claim is supported by context.
    faithfulness = supported_claims / total_claims
    """
    # Step 1: decompose answer into atomic claims
    decompose_prompt = (
        f"Break the following answer into individual atomic factual claims.\n"
        f"Return one claim per line.\n\nAnswer: {answer}"
    )
    claims_resp = llm.invoke(decompose_prompt)
    claims_text = getattr(claims_resp, "content", str(claims_resp))
    claims = [c.strip() for c in claims_text.strip().split("\n") if c.strip()]

    # Step 2: verify each claim against context
    context_str = "\n\n".join(contexts)
    supported = []
    for claim in claims:
        verify_prompt = (
            f"Context:\n{context_str}\n\n"
            f"Is the following claim supported by the context above? Answer YES or NO.\n"
            f"Claim: {claim}"
        )
        verdict = llm.invoke(verify_prompt)
        verdict_text = getattr(verdict, "content", str(verdict)).upper()
        supported.append("YES" in verdict_text)

    score = sum(supported) / len(claims) if claims else 0.0
    unsupported = [c for c, s in zip(claims, supported) if not s]
    return score, unsupported


# TruLens usage example
TRULENS_EXAMPLE = '''
from trulens_eval import Tru, TruChain, Feedback
from trulens_eval.feedback.provider import OpenAI

tru      = Tru()
provider = OpenAI()

# Define feedback functions
f_relevance   = Feedback(provider.relevance).on_input_output()
f_groundedness = Feedback(provider.groundedness_measure_with_cot_reasons)\
                    .on(context).on_output()

# Wrap your RAG chain
tru_chain = TruChain(
    my_rag_chain,
    app_id="advanced-rag-v1",
    feedbacks=[f_relevance, f_groundedness]
)

with tru_chain as recording:
    response = my_rag_chain.invoke({"question": "What is HNSW?"})

tru.get_leaderboard()   # view in dashboard: tru.run_dashboard()
'''
print("TruLens evaluation wrapper:")
print(TRULENS_EXAMPLE)

TruLens evaluation wrapper:

from trulens_eval import Tru, TruChain, Feedback
from trulens_eval.feedback.provider import OpenAI

tru      = Tru()
provider = OpenAI()

# Define feedback functions
f_relevance   = Feedback(provider.relevance).on_input_output()
f_groundedness = Feedback(provider.groundedness_measure_with_cot_reasons)                    .on(context).on_output()

# Wrap your RAG chain
tru_chain = TruChain(
    my_rag_chain,
    app_id="advanced-rag-v1",
    feedbacks=[f_relevance, f_groundedness]
)

with tru_chain as recording:
    response = my_rag_chain.invoke({"question": "What is HNSW?"})

tru.get_leaderboard()   # view in dashboard: tru.run_dashboard()



## 8. End-to-End Advanced RAG Pipeline

This section assembles all previous techniques into a single production-grade pipeline:

```
Documents
    │
    ▼
Hierarchical Chunking
    │
    ▼
Index: Vector Store (HNSW) + BM25
    │
    ▼  [at query time]
Query Transformation (HyDE or multi-query)
    │
    ▼
Hybrid Retrieval (dense + BM25 → RRF)
    │  top-50 candidates
    ▼
Reranking (cross-encoder or ColBERT)
    │  top-5 contexts
    ▼
Generation (LLM with context)
    │
    ▼
Evaluation (RAGAS)
```

### Design Decisions

| Decision | Options | Recommendation |
|---|---|---|
| Chunk size | 128 / 256 / 512 tokens | 256 child + 512 parent |
| Embedding model | `text-embedding-3-small`, `bge-large-en-v1.5` | BGE for open-source |
| Vector DB | Chroma, Qdrant, Weaviate, PGVector | Qdrant for production |
| Query transform | none / HyDE / multi-query | Multi-query for best recall |
| Reranker | cross-encoder / ColBERT / Cohere | BGE-reranker for open, Cohere for managed |
| Top-k candidates | 20-200 | 50 for most use cases |
| Final contexts | 3-10 | 5 is the sweet spot |

In [14]:
# End-to-end Advanced RAG Pipeline
import numpy as np
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field

@dataclass
class Document:
    page_content: str
    metadata: Dict = field(default_factory=dict)


class AdvancedRAGPipeline:
    """
    Full pipeline:
      hierarchical chunking → hybrid retrieval (BM25 + dense + RRF)
      → cross-encoder reranking → LLM generation
    """

    def __init__(
        self,
        documents: List[Document],
        llm,
        embed_fn,        # callable: List[str] -> np.ndarray
        reranker=None,   # optional cross-encoder
        top_k_retrieve: int = 20,
        top_k_rerank:   int = 5,
    ):
        self.llm             = llm
        self.embed_fn        = embed_fn
        self.reranker        = reranker
        self.top_k_retrieve  = top_k_retrieve
        self.top_k_rerank    = top_k_rerank

        self.docs            = documents
        self.doc_texts       = [d.page_content for d in documents]
        self.doc_embeddings  = embed_fn(self.doc_texts)     # (N, D)
        self.bm25            = build_bm25(self.doc_texts)

    # ── Retrieval ──────────────────────────────────────────────────────────────
    def _dense_retrieve(self, query_emb: np.ndarray, k: int) -> List[int]:
        norms  = np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True) + 1e-9
        normed = self.doc_embeddings / norms
        qn     = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        scores = normed @ qn
        return list(np.argsort(scores)[::-1][:k])

    def _bm25_retrieve(self, query: str, k: int) -> List[int]:
        scores = self.bm25.get_scores(query.lower().split())
        return list(np.argsort(scores)[::-1][:k])

    def _rrf_fuse(self, rankings: List[List[int]], k: int = 60) -> List[int]:
        scores: Dict[int, float] = {}
        for ranked in rankings:
            for rank, idx in enumerate(ranked):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
        return [idx for idx, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

    def retrieve(self, query: str) -> List[Document]:
        query_emb   = self.embed_fn([query])[0]
        dense_rank  = self._dense_retrieve(query_emb, self.top_k_retrieve)
        bm25_rank   = self._bm25_retrieve(query, self.top_k_retrieve)
        fused_rank  = self._rrf_fuse([dense_rank, bm25_rank])
        return [self.docs[i] for i in fused_rank[:self.top_k_retrieve]]

    # ── Reranking ──────────────────────────────────────────────────────────────
    def rerank(self, query: str, candidates: List[Document]) -> List[Document]:
        if self.reranker is None:
            return candidates[:self.top_k_rerank]
        texts  = [d.page_content for d in candidates]
        pairs  = [[query, t] for t in texts]
        scores = self.reranker.predict(pairs)
        ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in ranked[:self.top_k_rerank]]

    # ── Generation ─────────────────────────────────────────────────────────────
    def generate(self, query: str, contexts: List[Document]) -> str:
        context_str = "\n\n".join(
            f"[{i+1}] {doc.page_content}" for i, doc in enumerate(contexts)
        )
        prompt = (
            f"You are a helpful assistant. Use the provided context to answer the question.\n\n"
            f"Context:\n{context_str}\n\n"
            f"Question: {query}\n"
            f"Answer:"
        )
        response = self.llm.invoke(prompt)
        return getattr(response, "content", str(response))

    # ── Full pipeline ──────────────────────────────────────────────────────────
    def query(self, question: str) -> Dict[str, Any]:
        candidates = self.retrieve(question)
        contexts   = self.rerank(question, candidates)
        answer     = self.generate(question, contexts)
        return {
            "question":   question,
            "answer":     answer,
            "contexts":   [c.page_content for c in contexts],
            "n_retrieved":len(candidates),
            "n_reranked": len(contexts),
        }


# ── Demo with mock components ─────────────────────────────────────────────────
class MockEmbedFn:
    """Mock embedding: random stable vectors per text."""
    def __call__(self, texts: List[str]) -> np.ndarray:
        D = 16
        out = []
        for t in texts:
            rng = np.random.default_rng(hash(t[:50]) % (2**32))
            out.append(rng.standard_normal(D))
        return np.array(out)

class MockLLM:
    def invoke(self, prompt: str) -> str:
        return f"[Mock answer based on {len(prompt)} char prompt]"

sample_docs = [
    Document("RAG retrieves documents to augment LLM generation.",    {"source": "paper"}),
    Document("Hybrid search combines BM25 and dense retrieval.",       {"source": "blog"}),
    Document("Cross-encoder rerankers improve retrieval precision.",   {"source": "paper"}),
    Document("HNSW provides fast approximate nearest neighbours.",     {"source": "docs"}),
    Document("ColBERT uses per-token embeddings with MaxSim scoring.", {"source": "paper"}),
    Document("RAGAS evaluates faithfulness, relevancy, and recall.",   {"source": "blog"}),
]

pipeline = AdvancedRAGPipeline(
    documents=sample_docs,
    llm=MockLLM(),
    embed_fn=MockEmbedFn(),
    top_k_retrieve=4,
    top_k_rerank=2,
)

result = pipeline.query("How does hybrid search work?")
print("Pipeline result:")
print(f"  Question:    {result['question']}")
print(f"  Retrieved:   {result['n_retrieved']} candidates")
print(f"  Reranked to: {result['n_reranked']} contexts")
print(f"  Contexts used:")
for c in result['contexts']:
    print(f"    - {c}")
print(f"  Answer: {result['answer']}")

Pipeline result:
  Question:    How does hybrid search work?
  Retrieved:   4 candidates
  Reranked to: 2 contexts
  Contexts used:
    - ColBERT uses per-token embeddings with MaxSim scoring.
    - RAGAS evaluates faithfulness, relevancy, and recall.
  Answer: [Mock answer based on 252 char prompt]


In [15]:
# Putting it all together: multi-query expansion + hybrid + rerank

from typing import List, Dict, Any

def full_advanced_rag(
    question: str,
    pipeline: AdvancedRAGPipeline,
    query_variants: List[str],   # pre-generated (or from LLM)
) -> Dict[str, Any]:
    """
    Multi-query RAG-Fusion + hybrid retrieval + reranking.
    """
    all_queries = [question] + query_variants

    # Retrieve candidates for each query variant
    all_results: List[List[Document]] = []
    for q in all_queries:
        all_results.append(pipeline.retrieve(q))

    # RRF across all result sets
    doc_id_to_doc: Dict[str, Document] = {}
    rrf_scores:    Dict[str, float]    = {}
    k = 60

    for results in all_results:
        for rank, doc in enumerate(results):
            doc_id = doc.page_content[:60]
            doc_id_to_doc[doc_id] = doc
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)

    fused_docs = [
        doc_id_to_doc[did]
        for did, _ in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Rerank the fused candidates
    reranked = pipeline.rerank(question, fused_docs)

    # Generate
    answer = pipeline.generate(question, reranked)

    return {
        "question":      question,
        "query_variants":query_variants,
        "n_fused":       len(fused_docs),
        "n_reranked":    len(reranked),
        "answer":        answer,
        "contexts":      [d.page_content for d in reranked],
    }


variants = [
    "What are the components of hybrid retrieval?",
    "How do BM25 and dense vectors combine for search?",
]

result = full_advanced_rag(
    question="How does hybrid search work?",
    pipeline=pipeline,
    query_variants=variants,
)

print("Multi-query RAG-Fusion result:")
print(f"  Queries used:  {1 + len(result['query_variants'])}")
print(f"  After fusion:  {result['n_fused']} unique docs")
print(f"  After rerank:  {result['n_reranked']} docs")
print(f"  Contexts:")
for c in result['contexts']:
    print(f"    → {c}")

Multi-query RAG-Fusion result:
  Queries used:  3
  After fusion:  6 unique docs
  After rerank:  2 docs
  Contexts:
    → Hybrid search combines BM25 and dense retrieval.
    → RAGAS evaluates faithfulness, relevancy, and recall.


## Additional Learning Resources

### Foundational Papers

| Paper | arxiv | Key Contribution |
|---|---|---|
| RAG (Lewis et al., 2020) | [2005.11401](https://arxiv.org/abs/2005.11401) | Original RAG framework |
| RAPTOR | [2401.18059](https://arxiv.org/abs/2401.18059) | Tree-based hierarchical retrieval |
| Self-RAG | [2310.11511](https://arxiv.org/abs/2310.11511) | Reflection tokens for adaptive retrieval |
| CRAG | [2401.15884](https://arxiv.org/abs/2401.15884) | Corrective retrieval with web fallback |
| FLARE | [2305.06983](https://arxiv.org/abs/2305.06983) | Forward-looking active retrieval |
| HyDE | [2212.10496](https://arxiv.org/abs/2212.10496) | Hypothetical document embeddings |
| ColBERT | [2004.12832](https://arxiv.org/abs/2004.12832) | Late interaction reranking |
| ColBERTv2 | [2112.01488](https://arxiv.org/abs/2112.01488) | Improved ColBERT with distillation |
| SPLADE | [2107.05720](https://arxiv.org/abs/2107.05720) | Learned sparse representations |
| HippoRAG | [2405.14831](https://arxiv.org/abs/2405.14831) | Hippocampus-inspired KG + PPR |
| LightRAG | [2410.05779](https://arxiv.org/abs/2410.05779) | Graph + vector hybrid |

### Benchmarks

- **[ANN-Benchmarks](https://ann-benchmarks.com/)** reproducible ANN index comparison (recall vs. QPS)
- **[BEIR](https://github.com/beir-cellar/beir)** 18 heterogeneous retrieval datasets for zero-shot evaluation
- **[MTEB](https://huggingface.co/spaces/mteb/leaderboard)** embedding model leaderboard
- **[RAGAS](https://docs.ragas.io/)** end-to-end RAG evaluation

### Survey Papers

- *Retrieval-Augmented Generation for Large Language Models: A Survey* [2312.10997](https://arxiv.org/abs/2312.10997)
- *A Survey on RAG Meets LLMs* [2405.06211](https://arxiv.org/abs/2405.06211)

### Recommended Libraries

```bash
pip install langchain langchain-community langchain-openai
pip install sentence-transformers FlagEmbedding
pip install ragatouille          # ColBERT in 3 lines
pip install rank-bm25            # BM25 retrieval
pip install ragas                # RAG evaluation
pip install lancedb              # serverless vector DB
pip install weaviate-client      # Weaviate
pip install pymilvus             # Milvus
pip install cohere               # Cohere rerank API
pip install flashrank            # lightweight reranker
pip install deepeval             # LLM evaluation
```

### Key Takeaways

1. **Chunk carefully** hierarchical chunking (parent+child) beats fixed-size for most use cases
2. **Always use hybrid** BM25+dense+RRF consistently outperforms either alone (no tuning needed)
3. **Rerank before generation** a cross-encoder on top-50 candidates is cheap and reliable
4. **Transform queries** multi-query or HyDE meaningfully improves recall
5. **Evaluate with RAGAS** measure all four dimensions: faithfulness, relevancy, precision, recall
6. **Consider architecture** Self-RAG / CRAG / FLARE shine for production quality-gating